# 02 — PCY, association rules and a rule-based recommender

1. PCY on the toy example of notebook 01, showing buckets, bitmap and candidates
2. PCY on all 3.2M baskets: identical pairs and counts to A-Priori, and an experiment with the bucket-table size.
   Extension: the Multistage and Multihash refinements of PCY
3. Association rules (support, confidence, lift)
4. A rule-based next-basket recommender, tuned on the **validation** users and evaluated once on the **test** users
5. Do same-basket rules predict the *next* basket?

Needs notebook 01 run first (it caches the baskets and saves the user split), ~2 GB RAM, ~10 minutes.

In [1]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.data import Baskets, load_tables, load_baskets, next_basket_truth, user_split
from src.apriori import apriori_pairs, association_rules
from src.pcy import frequent_pairs, mix, BUCKET_MULTS, KEY_BASE, METHODS
from src.recommend import global_top, recent_products, recommend_rules, evaluate_at_k

ROOT = Path.cwd().parent
DERIVED = ROOT / "data" / "derived"
K = 10

## 1. PCY on the toy example

Same five baskets as in notebook 01, minimum support count 3, and only 9 buckets so the hashing is visible.
Pass 1 counts items and adds 1 to bucket $h(i,j)$ for **every** pair in every basket. Buckets with count ≥ 3 become 1-bits in the bitmap.
Pass 2 counts a pair only if both items are frequent **and** its bucket's bit is 1. Afterwards the exact counts are filtered by the threshold.

In [2]:
def make_baskets(lists):
    indptr = np.cumsum([0] + [len(b) for b in lists]).astype(np.int64)
    return Baskets(indptr, np.concatenate(lists).astype(np.int32), np.arange(len(lists), dtype=np.int32))

toy_lists = [[1, 2, 3], [1, 2, 5], [1, 3, 4], [2, 4], [1, 2, 3, 4]]
toy = make_baskets(toy_lists)
N_BUCKETS, MIN_COUNT = 9, 3

bucket = lambda a, b: mix(np.int64(a * KEY_BASE + b), BUCKET_MULTS[0]) % N_BUCKETS  # the hash PCY uses
item_counts = np.bincount(toy.items)
pair_counts = pd.Series([(a, b) for bs in toy_lists for i, a in enumerate(sorted(bs)) for b in sorted(bs)[i + 1:]]).value_counts()
table = pd.DataFrame({"pair": pair_counts.index, "count": pair_counts.to_numpy()})
table["bucket"] = [bucket(a, b) for a, b in table["pair"]]
bucket_counts = table.groupby("bucket")["count"].sum()
table["bucket count"] = table["bucket"].map(bucket_counts)
table["bitmap"] = (table["bucket count"] >= MIN_COUNT).astype(int)
table["both items frequent"] = [item_counts[a] >= MIN_COUNT and item_counts[b] >= MIN_COUNT for a, b in table["pair"]]
table["counted in pass 2"] = table["both items frequent"] & (table["bitmap"] == 1)
print("item counts:", {i: int(c) for i, c in enumerate(item_counts) if c})
table.sort_values(["bucket", "pair"]).reset_index(drop=True)

item counts: {1: 4, 2: 4, 3: 3, 4: 3, 5: 1}


,pair,count,bucket,bucket count,bitmap,both items frequent,counted in pass 2
0,"(2, 3)",2,4,2,0,True,False
1,"(2, 5)",1,5,3,1,False,False
2,"(3, 4)",2,5,3,1,True,True
3,"(1, 2)",3,6,6,1,True,True
4,"(1, 3)",3,6,6,1,True,True
5,"(1, 4)",2,7,3,1,True,True
6,"(1, 5)",1,7,3,1,False,False
7,"(2, 4)",2,8,2,0,True,False


In [3]:
pcy_pairs, stats = frequent_pairs(toy, MIN_COUNT, "pcy", n_buckets=N_BUCKETS)
ap_pairs, _ = apriori_pairs(toy, MIN_COUNT)
print(f"PCY counted {stats['candidate pairs']} candidate pairs in pass 2 (A-Priori: all 6 pairs of frequent items)")
print(pcy_pairs)
assert table["counted in pass 2"].sum() == stats["candidate pairs"]
as_set = lambda df: set(map(tuple, df[["item_a", "item_b", "count"]].to_numpy().tolist()))
assert as_set(pcy_pairs) == as_set(ap_pairs) == {(1, 2, 3), (1, 3, 3)}

PCY counted 4 candidate pairs in pass 2 (A-Priori: all 6 pairs of frequent items)
   item_a  item_b  count
0       1       2      3
1       1       3      3


## 2. PCY on all baskets

**Extension: Multistage and Multihash.** Multistage adds a pass that rehashes the pairs surviving PCY's bitmap with a second, independent hash function, and counts only pairs passing both bitmaps.
Multihash instead uses two half-size hash tables in pass 1. Both are in `src/pcy.py`, next to PCY.

First a check that all three return exactly A-Priori's pairs and counts at the same threshold.

In [4]:
t = time.time()
tables = load_tables()
orders, prior, products = tables["orders"], tables["prior"], tables["products"]
name = products.set_index("product_id")["product_name"]
baskets = load_baskets()
N = len(baskets)
truth = next_basket_truth(orders, tables["train"])
split = user_split(truth)
print(f"{N:,} baskets; {len(split['validation']):,} validation and {len(split['test']):,} test users  ({time.time() - t:.0f}s)")

min_count = int(1e-4 * N)
reference, _ = apriori_pairs(baskets, min_count)
for method in METHODS:
    pairs, _ = frequent_pairs(baskets, min_count, method)
    assert as_set(pairs) == as_set(reference), method
print(f"PCY, Multistage and Multihash = A-Priori at support 1e-4: {len(reference):,} identical pairs and counts")

3,214,874 baskets; 13,368 validation and 65,444 test users  (5s)


PCY, Multistage and Multihash = A-Priori at support 1e-4: 57,035 identical pairs and counts


### How many buckets does PCY need?

At support $2\cdot10^{-5}$ (64 baskets), for several bucket-table sizes: the share of buckets that pass the threshold,
the number of candidate pairs counted in the last pass, and its memory (bitmap bits plus 12 bytes per candidate pair).
With too few buckets almost every bucket is frequent and PCY degenerates to counting every pair of frequent items.

In [5]:
min_count = int(2e-5 * N)
rows, found = [], set()
for log_b in (18, 20, 22, 24):
    for method in METHODS:
        pairs, stats = frequent_pairs(baskets, min_count, method, n_buckets=1 << log_b)
        found.add(len(pairs))
        rows.append({"buckets": f"2^{log_b}", **stats})
assert len(found) == 1, "every setting must find the same frequent pairs"
print(f"every setting finds the same {found.pop():,} frequent pairs")
pd.DataFrame(rows)[["buckets", "method", "frequent buckets", "candidate pairs", "pass-1 bucket MB", "last-pass MB", "seconds"]].round(
    {"pass-1 bucket MB": 1, "last-pass MB": 2, "seconds": 2})

every setting finds the same 428,017 frequent pairs


,buckets,method,frequent buckets,candidate pairs,pass-1 bucket MB,last-pass MB,seconds
0,2^18,pcy,100.0%,34744756,1.0,416.97,26.91
1,2^18,multistage,100.0% / 100.0%,34744756,1.0,417.00,23.26
2,2^18,multihash,100.0% / 100.0%,34744756,1.0,416.97,21.86
3,2^20,pcy,98.4%,34341333,4.2,412.23,21.72
4,2^20,multistage,98.4% / 94.9%,33065214,4.2,397.04,20.96
5,2^20,multihash,100.0% / 100.0%,34744756,4.2,417.07,21.75
6,2^22,pcy,19.3%,7966971,16.8,96.13,14.05
7,2^22,multistage,19.3% / 11.3%,1424440,16.8,18.14,13.88
8,2^22,multihash,56.9% / 56.0%,13320498,16.8,160.37,16.08
9,2^24,pcy,3.0%,1634823,67.1,21.72,15.45


## 3. Association rules on all baskets

Every frequent pair $\{a,b\}$ gives two rules $a\to b$ and $b\to a$:

- support $= \text{count}(a,b)/N$
- confidence $= \text{count}(a,b)/\text{count}(a)$, the estimated $P(b \mid a)$ in the same basket
- lift $= \text{confidence}/\text{support}(b)$: how much more often $b$ appears with $a$ than on its own

In [6]:
SUPPORTS = (1e-4, 5e-5, 2e-5)
item_counts = np.bincount(baskets.items)
rule_sets = {}
for s in SUPPORTS:
    t = time.time()
    pairs, stats = frequent_pairs(baskets, int(s * N), "multistage")
    rule_sets[s] = association_rules(pairs, item_counts, N)
    print(f"support {s:g}: {len(pairs):,} frequent pairs, {len(rule_sets[s]):,} rules  ({time.time() - t:.0f}s)")
del baskets

support 0.0001: 57,035 frequent pairs, 114,070 rules  (8s)


support 5e-05: 139,581 frequent pairs, 279,162 rules  (9s)


support 2e-05: 428,017 frequent pairs, 856,034 rules  (14s)


In [7]:
rules = rule_sets[1e-4]
named = lambda df: df.assign(antecedent=name[df["antecedent"]].to_numpy(), consequent=name[df["consequent"]].to_numpy())
cols = ["antecedent", "consequent", "support", "confidence", "lift"]
print("Highest confidence (support >= 1e-4):")
print(named(rules.sort_values("confidence", ascending=False).head(8))[cols].round(4).to_string(index=False))
print("\nPasta -> sauce rules:")
pasta = products.loc[products["aisle"] == "dry pasta", "product_id"]
sauce = products.loc[products["aisle"] == "pasta sauce", "product_id"]
ps = rules[rules["antecedent"].isin(pasta) & rules["consequent"].isin(sauce)]
print(named(ps.sort_values("confidence", ascending=False).head(5))[cols].round(4).to_string(index=False))

Highest confidence (support >= 1e-4):
                                                       antecedent                                          consequent  support  confidence      lift
Oh My Yog! Organic Wild Quebec Blueberry Cream Top Yogurt & Fruit Oh My Yog! Pacific Coast Strawberry Trilayer Yogurt   0.0005      0.6611  743.9013
                                Grain Free Turkey Canned Cat Food                 Grain Free Chicken Formula Cat Food   0.0001      0.6429 1132.4409
                       Organic Cottage Cheese Blueberry Acai Chia    Organic Strawberry Chia Lowfat 2% Cottage Cheese   0.0002      0.6162 1696.0931
                         Organic Cashew Nondairy Blueberry Yogurt           Organic Nondairy Strawberry Cashew Yogurt   0.0001      0.6161 2167.1523
                                                        Blueberry                                              Banana   0.0002      0.5733    3.9004
                                     Almond Milk Blueberry Yogurt   

In [8]:
rules.to_csv(DERIVED / "rules_support_1e-4.csv", index=False)
print(f"saved {len(rules):,} rules for the other notebooks")

saved 114,070 rules for the other notebooks


## 4. Rule-based recommender

For each user, every rule $a\to b$ whose antecedent $a$ is among the products of their most recent prior order(s) votes for $b$;
the 10 products with the highest summed votes are recommended, and popular products fill any empty positions.
Products the user bought before are allowed, because most of a next basket is repeat purchases.

**Tuning on the validation users** (never on test): the rule support threshold, how many recent orders form the input,
and whether votes are confidences or lifts. The primary metric is hit-rate@10 (share of users with at least one correct product); we also report precision and recall@10.

In [9]:
top = global_top(prior, K)
val_truth = {u: truth[u] for u in split["validation"]}
histories = {n: recent_products(orders, prior, split["validation"], n) for n in (1, 3, 5)}

rows = []
for s, r in rule_sets.items():
    for n_last, hist in histories.items():
        for score in ("confidence", "lift"):
            recs = recommend_rules(hist, r, split["validation"], K, top, score=score)
            rows.append({"support": s, "recent orders": n_last, "votes": score, **evaluate_at_k(recs, val_truth, K)})
tuning = pd.DataFrame(rows).drop(columns="users").sort_values("hit_rate@10", ascending=False, ignore_index=True)
tuning.round({"precision@10": 4, "recall@10": 4, "hit_rate@10": 4})

,support,recent orders,votes,precision@10,recall@10,hit_rate@10
0,0.00002,3,confidence,0.0846,0.0863,0.5054
1,0.00002,5,confidence,0.0843,0.0858,0.5043
2,0.00002,1,confidence,0.0839,0.0839,0.5020
3,0.00005,5,confidence,0.0834,0.0850,0.5016
4,0.00005,3,confidence,0.0833,0.0846,0.4999
5,0.00010,5,confidence,0.0825,0.0837,0.4960
6,0.00005,1,confidence,0.0828,0.0825,0.4959
7,0.00010,3,confidence,0.0825,0.0834,0.4950
8,0.00010,1,confidence,0.0815,0.0806,0.4885
9,0.00010,3,lift,0.0649,0.0647,0.3877


In [10]:
best = tuning.iloc[0]
print(f"chosen on validation: support {best['support']:g}, last {best['recent orders']} orders, votes = {best['votes']}")

chosen on validation: support 2e-05, last 3 orders, votes = confidence


## Final evaluation on the test users (used once)

In [11]:
test_users = split["test"]
test_truth = {u: truth[u] for u in test_users}
hist = recent_products(orders, prior, test_users, int(best["recent orders"]))
rule_recs = recommend_rules(hist, rule_sets[best["support"]], test_users, K, top, score=best["votes"])
pd.Series(evaluate_at_k(rule_recs, test_truth, K)).round(4)

precision@10        0.0849
recall@10           0.0874
hit_rate@10         0.5062
users           65444.0000
dtype: float64

In [12]:
pd.DataFrame([(u, rank + 1, p) for u, ps in rule_recs.items() for rank, p in enumerate(ps)],
             columns=["user_id", "rank", "product_id"]).to_csv(DERIVED / "rule_recommendations_test.csv", index=False)
u = test_users[3]
print("last order:     ", "; ".join(name[hist.loc[hist["user_id"] == u, "product_id"]]))
print("recommended:    ", "; ".join(f"{name[p]}{' ✓' if p in truth[u] else ''}" for p in rule_recs[u]))
print("actual order:   ", "; ".join(name[list(truth[u])]))

last order:      Original Hummus; Prosciutto di San Daniele; Red Peppers; Cantaloupe; Organic Chicken & Apple Sausage; Ground Turkey Breast; Organic Spinach & Feta Dinner Sausage; Beef Loin New York Strip Steak; Asparagus; Cucumber & Garlic  Tzatziki; Goat Cheese Log; Uncured Slow Cooked Ham; Applewood Smoked Turkey Breast; Feta Cheese Crumbles; Organic Whole String Cheese; Non Fat Raspberry Yogurt; Extra Virgin Olive Oil; Mild Salsa; Veri Veri Teriyaki Marinade & Sauce; Bunched Cilantro; Flat Parsley, Bunch; Parsley, Italian (Flat), New England Grown; Sage; Organic Poultry Blend Herbs; Organic Baby Spinach Salad; Mache Rosettes French Salad Mix; Organic Grape Tomatoes; Organic Red On the Vine Tomato; Strawberries; Sharp Cheddar Cheese; New York Style Nova Lox; Organic Chicken Stock; Organic Snipped Green Beans; Organic Zucchini; Organic Small Bunch Celery; Carrots; Organic Russet Potato; Small Hass Avocado; Organic Large Green Asparagus; Organic Red Cabbage; I Heart Baby Kale; Organic

## 5. Same-basket rules vs. the next basket

A rule's confidence estimates $P(b \mid a)$ **in the same basket**. The recommender instead needs $P(b$ in the *next* order $\mid a$ in the last order$)$.
For the validation users we measure the latter for every rule that fires, and compare it with the rule's confidence,
separately for consequents the user had bought before and products new to them.

In [13]:
r = rule_sets[best["support"]]
r = r.sort_values("confidence", ascending=False).groupby("antecedent").head(20)
fired = recent_products(orders, prior, split["validation"], 1).merge(r, left_on="product_id", right_on="antecedent")
owned = set(map(tuple, recent_products(orders, prior, split["validation"], 100)[["user_id", "product_id"]].to_numpy()))
fired["in next order"] = [b in truth[u] for u, b in zip(fired["user_id"], fired["consequent"])]
fired["bought before"] = [(u, b) in owned for u, b in zip(fired["user_id"], fired["consequent"])]
fired["confidence bin"] = pd.cut(fired["confidence"], [0, 0.05, 0.1, 0.2, 0.3, 1])
summary = fired.groupby(["confidence bin", "bought before"], observed=True).agg(
    rules_fired=("in next order", "size"), mean_confidence=("confidence", "mean"), next_order_rate=("in next order", "mean"))
print(f"{len(fired):,} (user, rule) firings; correlation of confidence with next-order hit: "
      f"{np.corrcoef(fired['confidence'], fired['in next order'])[0, 1]:.3f}")
print(f"next-order hit rate: consequent bought before {fired.loc[fired['bought before'], 'in next order'].mean():.3f}, "
      f"new to the user {fired.loc[~fired['bought before'], 'in next order'].mean():.3f}")
summary.round(3)

2,221,296 (user, rule) firings; correlation of confidence with next-order hit: 0.166
next-order hit rate: consequent bought before 0.241, new to the user 0.019


rules_fired  mean_confidence  next_order_rate
confidence bin bought before                                               
(0.0, 0.05]    False               294158            0.041            0.010
               True                 52852            0.042            0.181
(0.05, 0.1]    False               896317            0.070            0.018
               True                293216            0.072            0.198
(0.1, 0.2]     False               344216            0.137            0.027
               True                207800            0.142            0.274
(0.2, 0.3]     False                61083            0.235            0.039
               True                 57736            0.237            0.376
(0.3, 1.0]     False                 6166            0.348            0.038
               True                  7752            0.354            0.405

**Conclusions.**

- **Tuning.** Votes by confidence beat votes by lift by a wide margin: lift favours rare niche pairs (e.g. two flavours of the same yogurt brand), confidence favours products that are bought often.
  A lower support threshold (more rules) and the last 3 orders as input are best; the differences between those settings are small.
- **Test.** On the 65,444 test users at least one of the 10 recommended products is in the next order for 50.6% of users (precision@10 0.085). The predictions are saved in `data/derived/rule_recommendations_test.csv` for the comparison with the baselines.
- **Same basket is not next basket.** A rule's confidence is a poor estimate of next-order probability. When the consequent is *new* to the user, it appears in the next order only 1–4% of the time even for rules with confidence above 0.3.
  When the user has bought it before, the rate is 18–41%, mostly because of the user's own history rather than the rule.
- So same-basket rules alone are a weak next-basket recommender: their signal is mostly the user's own repeat purchases. Their value is in suggesting products a user has *not* bought, where the user's history cannot help.